# Intelligent Agents: Reflex-Based Agents for GridHunt (Multi-agent Environment)

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [23]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Bug fix (found via verbose debugging, see debugging_in_notebooks.ipynb):
        # if the monster's move lands it on a hunter's square, that hunter caught
        # it right there. This must be checked BEFORE asking the hunters to act,
        # otherwise a hunter agent computing distance = 0 could try to return an
        # invalid 'stay' action and crash move().
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster (monster moved onto Hunter 1) in {step} steps!")
            return ("1", step)
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster (monster moved onto Hunter 2) in {step} steps!")
            return ("2", step)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)


I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [24]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [25]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [26]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [27]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('south')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [28]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[2 0]'; Hunter 2 is at '[2 1]'; Monster is at '[0 2]'

..M..
.....
12...
.....
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'north'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[3 0]'; Hunter 2 is at '[1 1]'; Monster is at '[0 2]'

..M..
.2...
.....
1....
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'west'
Monster chose action 'stay'

=====

Step 2:
Hunter 1 is at '[4 0]'; Hunter 2 is at '[1 0]'; Monster is at '[0 2]'

..M..
2....
.....
.....
1....
Hunter 1 chose action 'east'
Hunter 2 chose action 'south'
Monster chose action 'stay'

=====

Step 3:
Hunter 1 is at '[4 1]'; Hunter 2 is at '[2 0]'; Monster is at '[0 2]'

..M..
.....
2....
.....
.1...
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[2 0]'; Hunter 2 is at '[2 1]'; Monster is at '[0 2]'

..M..
.....
12...
.....
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'teleport'
Monst

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [29]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('1', 40), ('2', 73), ('M', nan), ('M', nan), ('M', nan), ('1', 6), ('M', nan), ('M', nan), ('1', 26), ('M', nan), ('M', nan), ('M', nan), ('1', 19), ('M', nan), ('M', nan), ('1', 18), ('M', nan), ('1', 82), ('2', 13), ('M', nan), ('M', nan), ('M', nan), ('1', 55), ('M', nan), ('1', 5), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 87), ('M', nan), ('M', nan), ('M', nan), ('1', 65), ('1', 37), ('1', 99), ('2', 86), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 20), ('1', 54), ('M', nan), ('M', nan), ('1', 9), ('M', nan), ('1', 61), ('1', 59), ('2', 37), ('1', 63), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 22), ('1', 40), ('2', 69), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 52), ('2', 48), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 89), ('1', 27), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 3), ('2', 15), ('M', nan), ('1', 59), ('2', 48), ('M', nan), ('1', 39), ('M', nan), ('1',

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [30]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 26
Hunter 2 won: 15
Monster 1 won: 59


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [31]:
rng = np.random.default_rng()

# ----------------------------------------------------------------------
# Chiến lược 1: không lưu lại state, chỉ thấy vị trí của mình + vị trí quái vật).
# Thợ săn luôn di chuyển theo trục (hàng hoặc cột) đang xa lẹch quái vật hơn (dùng Manhattan distance).
# ----------------------------------------------------------------------
def greedy_hunter_agent(hunter_position, monster_position, verbose=False):
    dr = monster_position[0] - hunter_position[0]
    dc = monster_position[1] - hunter_position[1]

    if abs(dr) >= abs(dc):
        action = "south" if dr > 0 else "north"
    else:
        action = "east" if dc > 0 else "west"

    if verbose:
        print(f"[greedy] hunter={tuple(hunter_position)} monster={tuple(monster_position)} "
              f"dr={dr} dc={dc} -> action='{action}'")
    return action


# ----------------------------------------------------------------------
# Chien luoc 2: Dùng Teleport khi ở quá xa quái vật
# Khi khoảng cách Manhattan còn lại quá lớn, thợ săn thử teleport (xác suất 30%) để rút ngắn quảng đường thay vì phải di chuyển từng bước.
# ----------------------------------------------------------------------
def smart_hunter_agent(hunter_position, monster_position, arena_size=30, verbose=False):
    manhattan_dist = abs(hunter_position[0] - monster_position[0]) + abs(hunter_position[1] - monster_position[1])

    if manhattan_dist > arena_size * 0.7 and rng.random() < 0.3:
        if verbose:
            print(f"[smart] dist={manhattan_dist} > 70% cua {arena_size} -> teleport")
        return "teleport"

    dr = monster_position[0] - hunter_position[0]
    dc = monster_position[1] - hunter_position[1]

    # Bug fix: ban goc dung "abs(dr) > abs(dc)" (dau ">" chat) nen khi
    # dr == dc == 0 ham roi vao nhanh "return 'stay'" -> khong hop le. Dung
    # ">=" de luon tra ve huong di chuyen hop le.
    if abs(dr) >= abs(dc):
        action = "south" if dr > 0 else "north"
    else:
        action = "east" if dc > 0 else "west"

    if verbose:
        print(f"[smart] hunter={tuple(hunter_position)} monster={tuple(monster_position)} "
              f"dist={manhattan_dist} dr={dr} dc={dc} -> action='{action}'")
    return action


# Kiểm tra thử hành động của từng chiến lược.
print("Hành động của Greedy Hunter khi hunter=[2,3], monster=[5,1]:",
      greedy_hunter_agent([2, 3], [5, 1], verbose=True))
print("Hành động của Smart Hunter khi hunter=[2,3], monster=[19,18] (arena=20):",
      smart_hunter_agent([2, 3], [19, 18], arena_size=20, verbose=True))


[greedy] hunter=(2, 3) monster=(5, 1) dr=3 dc=-2 -> action='south'
Hành động của Greedy Hunter khi hunter=[2,3], monster=[5,1]: south
[smart] dist=32 > 70% cua 20 -> teleport
Hành động của Smart Hunter khi hunter=[2,3], monster=[19,18] (arena=20): teleport


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [32]:
# Your experimentation code

def run_exp(h1_fn, h2_fn, monster_fn, arena_size=30, num_runs=100, max_steps=100, verbose=False):
    results = {"1": 0, "2": 0, "M": 0}
    steps_list = []
    for i in range(num_runs):
        winner, steps = arena_environment(
            h1_fn, h2_fn, monster_fn,
            n=arena_size, max_steps=max_steps, visualize=False
        )
        results[winner] += 1
        if not np.isnan(steps):
            steps_list.append(steps)
        if verbose:
            print(f"run {i+1}/{num_runs}: winner='{winner}', steps={steps}")

    avg_steps = np.mean(steps_list) if steps_list else np.nan
    return results, avg_steps


# 1. Mô phỏng thử nghiệm 1 ván đấu
print("--- 1. Mô phỏng (arena 20x20) ---")
arena_environment(greedy_hunter_agent, simple_randomized_hunter_agent_function,
                   monster_agent_function_simple, n=20, max_steps=15, visualize=True)

# ------------------------------------------------------------------
# 2. Thí nghiệm chính:
#    A. Greedy(1) vs Random(2), 20x20, 100 ván   -> đếm số lần thắng
#    B. Greedy(1) vs Random(2), 30x30, 100 ván   -> đếm số lần thắng
#    C. Greedy(1) vs Smart+Teleport(2), 30x30    -> so sánh 2 chiến lược
# ------------------------------------------------------------------
print("\n--- 2. Kết quả thí nghiệm (100 ván sau khi cấu hình) ---")
print(f"{'Kịch bản':<42} | {'Hunter1':<8} | {'Hunter2':<8} | {'Monster':<8} | {'TB buoc':<8}")
print("-" * 86)

scenarios = [
    ("A: Greedy(1) vs Random(2), 20x20",
     greedy_hunter_agent, simple_randomized_hunter_agent_function, monster_agent_function_simple, 20),
    ("B: Greedy(1) vs Random(2), 30x30",
     greedy_hunter_agent, simple_randomized_hunter_agent_function, monster_agent_function_simple, 30),
    ("C: Greedy(1) vs Smart+Teleport(2), 30x30",
     greedy_hunter_agent, lambda h, m: smart_hunter_agent(h, m, arena_size=30), monster_agent_function_simple, 30),
]

exp_results = {}
for label, h1, h2, mfn, size in scenarios:
    res, avg_steps = run_exp(h1, h2, mfn, arena_size=size, num_runs=100)
    exp_results[label] = (res, avg_steps)
    print(f"{label:<42} | {res['1']:<8} | {res['2']:<8} | {res['M']:<8} | {avg_steps:<8.1f}")


--- 1. Mô phỏng (arena 20x20) ---
Step 0:
Hunter 1 is at '[ 1 16]'; Hunter 2 is at '[10  3]'; Monster is at '[13 12]'

....................
................1...
....................
....................
....................
....................
....................
....................
....................
....................
...2................
....................
....................
............M.......
....................
....................
....................
....................
....................
....................
Hunter 1 chose action 'south'
Hunter 2 chose action 'west'
Monster chose action 'stay'


Step 1:
Hunter 1 is at '[ 2 16]'; Hunter 2 is at '[10  2]'; Monster is at '[13 12]'

....................
....................
................1...
....................
....................
....................
....................
....................
....................
....................
..2.................
....................
....................
............M.

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

> **Chiến lược:** Sử dụng simple-reflex agent (`greedy_hunter_agent`):
> so sánh khoảng cách giữa các hàng (`dr`) và cột (`dc`) tới vị trí của quái vật, luôn di chuyển
> chuyển theo trục đang lệch nhiều hơn (rut ngắn Manhattan distance nhanh
> nhất). Chiến lược này hiệu quả percept cho biết chính xác vị trí mỗi bước của quái vật,
> nên không cần lập kế hoạch hay tìm kiếm - chỉ cần luôn bước gần về phía quái vật.
> `smart_hunter_agent` mở rộng thêm một bước: khi còn xa quái vật (khoảng hơn 70% kích thước đấu trường),
> nó sẽ thử `teleport` với xác suất là 30%.
>
> **Về teleport:** Chỉ dùng khi khoảng cách còn lại quá lớn, và với xác suất là 30%. 
> Lý do : teleport hunter đến 1 ô ngẫu nhiên trong đấu trường - có thể giúp rút ngắn quảng đường dài 
> nhanh chóng, nhưng cũng có thể đưa hunter ra xa quái vật hơn. 
> Vì vậy nó chỉ đang bước một cách ngẫu khi đã ở rất xa (lúc đó di chuyển cũng tốt rất nhiều bước
> nên rủi ro là chấp nhận được), còn khi ở gần thì luôn di chuyển về hướng quái vật 
> vì lúc đó lựa chọn đã chắc chắn hơn.
>
> **Ảnh hưởng của kích thước đấu trường:** Như đã thấy ở thí nghiệm A và B, đấu 
> trường càng lớn thì số bước để bắt được quái vật càng tăng (do
> khoảng cách ban đầu lớn hơn và quái vật có nhiều không giam để chạy hơn)
> và tỉ lệ quái vật sống sót cũng tăng theo kích thước đấu trường.


The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

> Quái vật hiện tại (`monster_agent_function_simple`) chỉ di chuyển ngẫu
> nhiên và phần lớn đứng yên (`p=0.5` cho "stay"), nên nó hoàn toàn không
> phản ứng với vị trí hunter. Để quái vật trốn nhanh hơn mà khong qua nhanh, 
> có thể biến nó thành 1 **model-based reflex agent**:
> với môi trường di chuyển có thể bao gồm "stay", tính khoảng cách tới
> hunter gần nhất, rồi chọn hướng làm khoảng cách đo lớn nhất.
> Để không quá nhanh: chỉ thực sự di chuyển khi có hunter trong một bán kính
> nguy hiểm nào đó (ví dụ Manhattan distance < 5), còn lại vẫn đứng yên
> phần lớn thời gian như bản gốc; hoặc giới hạn xác suất di chuyển thành
> công (ví dụ 70%) để mô phỏng quái vật chậm hơn hunter một chút.


# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.

## Đề xuất: biến môi trường cạnh tranh thành môi trường hợp tác (chia sẻ vị trí)

Hiện tại cả hai hunter đã nhận được vị trí chính xác của quai vật (sensor
toàn cục trong PEAS), nhưng không biết vị trí của nhau - mỗi hàm agent chỉ
nhận `(vi_tri_cua_chinh_minh, vi_tri_quai_vat)`. Vì vậy cả hai hunter chạy
cùng một chiến lược độc lập sẽ có xu hướng đi theo gần như cùng một đường
tới quai vật, gây trùng lặp nỗ lực thay vì phối hợp bao vây.

**Đề xuất**: tạo một class `HunterTeam` giữ chung một "blackboard" lưu vị trí mới nhất của
cả 2 hunter. Hai method của cùng một đối tượng (`team.hunter_1_act`,
`team.hunter_2_act`) được truyền vào `arena_environment` y hệt hai hàm
thường, nhưng vì chia sẻ `self`, chúng tự động biết vị trí của nhau mà
không cần sửa đổi `arena_environment` hay thêm percept nào cả. Từ đó có thể
phân vai trò động: hunter nào đang gần quái vật hơn thực hiện đuổi trực
tiếp, hunter còn lại đi đón đầu hướng quai vật có thể chạy trốn, ước lượng từ vị trí của hunter đuổi.

In [33]:
# 2 hunter chia sẻ trang thai qua chung mot class Python
class HunterTeam:
    def __init__(self, arena_size=30, verbose=False):
        self.arena_size = arena_size
        self.verbose = verbose
        self.pos = {1: None, 2: None}  # "blackboard" chung cho ca hai hunter

    def _direction(self, dr, dc):
        if abs(dr) >= abs(dc):
            return "south" if dr > 0 else "north"
        else:
            return "east" if dc > 0 else "west"

    def _act(self, me, hunter_pos, monster_pos):
        self.pos[me] = np.array(hunter_pos)
        other = self.pos[3 - me]  # vị trí hunter còn lại, đọc từ blackboard chung

        if other is None:
            dr = monster_pos[0] - hunter_pos[0]
            dc = monster_pos[1] - hunter_pos[1]
            return self._direction(dr, dc)

        my_dist = abs(hunter_pos[0]-monster_pos[0]) + abs(hunter_pos[1]-monster_pos[1])
        other_dist = abs(other[0]-monster_pos[0]) + abs(other[1]-monster_pos[1])

        if my_dist <= other_dist:
            role = "chaser"
            dr = monster_pos[0] - hunter_pos[0]
            dc = monster_pos[1] - hunter_pos[1]
        else:
            role = "interceptor"
            flee_r = np.sign(monster_pos[0] - other[0])
            flee_c = np.sign(monster_pos[1] - other[1])
            target = np.clip(np.array([monster_pos[0] + flee_r*2, monster_pos[1] + flee_c*2]),
                              0, self.arena_size - 1)
            dr, dc = target[0] - hunter_pos[0], target[1] - hunter_pos[1]
            if dr == 0 and dc == 0:
                dr = monster_pos[0] - hunter_pos[0]
                dc = monster_pos[1] - hunter_pos[1]

        if self.verbose:
            print(f"[team H{me}] role={role} my_dist={my_dist} other_dist={other_dist}")
        return self._direction(dr, dc)

    def hunter_1_act(self, hunter_pos, monster_pos):
        return self._act(1, hunter_pos, monster_pos)

    def hunter_2_act(self, hunter_pos, monster_pos):
        return self._act(2, hunter_pos, monster_pos)


# So sánh: 2 hunter độc lập (không biết vị trí của nhau) vs 2 hunter hợp tác
for size in (20, 30):
    team_wins, team_steps = 0, []
    for _ in range(100):
        team = HunterTeam(arena_size=size) 
        w, s = arena_environment(team.hunter_1_act, team.hunter_2_act,
                                  monster_agent_function_simple, n=size, max_steps=100)
        if w in ("1", "2"):
            team_wins += 1
            team_steps.append(s)

    indep_res, indep_avg = run_exp(greedy_hunter_agent, greedy_hunter_agent,
                                    monster_agent_function_simple, arena_size=size, num_runs=100)
    indep_wins = indep_res["1"] + indep_res["2"]

    print(f"Đấu trường {size}x{size}:")
    print(f"  2 hunter độc lập (greedy, không biết vị trí của nhau) -> {indep_wins}/100 thắng, avg steps={indep_avg:.1f}")
    print(f"  2 hunter hợp tác (chia sẻ vị trí qua HunterTeam)  -> {team_wins}/100 thắng, avg steps={np.mean(team_steps):.1f}")


Đấu trường 20x20:
  2 hunter độc lập (greedy, không biết vị trí của nhau) -> 100/100 thắng, avg steps=10.1
  2 hunter hợp tác (chia sẻ vị trí qua HunterTeam)  -> 100/100 thắng, avg steps=8.8
Đấu trường 30x30:
  2 hunter độc lập (greedy, không biết vị trí của nhau) -> 100/100 thắng, avg steps=15.0
  2 hunter hợp tác (chia sẻ vị trí qua HunterTeam)  -> 100/100 thắng, avg steps=14.2


> **Nhận xét (dựa trên kết quả chạy thực tế ở trên):** Với một quai vật chỉ
> di chuyển ngẫu nhiên/đứng yên (không chủ động trốn), lợi ích của việc chia
> sẻ vị trí và phân vai trò động (chaser/interceptor) thường không chênh
> lệch nhiều so với 2 hunter greedy độc lập - vì khi hunter đã có percept
> toàn cục, chiến lược greedy đơn giản đã rất gần tối ưu rồi. Lợi ích của
> hợp tác sẽ rõ rệt hơn trong các tình huống: (1) quai vật thật sự biết trốn
> (nhìn xa hơn 1 bước), khi này "chặn đường thoát" mới có giá trị; (2) đấu
> trường rất lớn với nhiều hơn 2 hunter, lúc đó chia vùng trách nhiệm sẽ
> giảm trùng lặp nỗ lực rõ rệt hơn; (3) heuristic đón đầu được tính toán kỹ
> hơn thay vì chỉ ngoại suy +2 ô đơn giản như ở đây.